In [22]:
import os
import sys
import json
import asyncio
import importlib

# 1. Thêm root dev_llm_service vào sys.path
sys.path.append(os.path.abspath(".."))

# 2. Reload các module RAG & Agent để luôn nhận code mới nhất
import app.core.config as config_module
import app.ai.rag.embedding.service as embed_module
import app.ai.rag.retrieval.retriever as ret_module
import app.ai.agent.agentic_rag.tools.search_policy_docs_tool as search_tool_module
import app.ai.agent.agentic_rag.graph.graph as rag_graph_module
import app.ai.orchestration.graph as orch_graph_module

importlib.reload(config_module)
importlib.reload(embed_module)
importlib.reload(ret_module)
importlib.reload(search_tool_module)
importlib.reload(rag_graph_module)
importlib.reload(orch_graph_module)

from app.core.config import settings
from app.routers.dependencies import get_embedding_service, get_vector_retriever, get_chat_model
from app.ai.agent.agentic_rag.tools import (
    search_policy_and_manual_docs,
    vector_search_tool,
    get_document_metadata,
    list_policy_categories,
    set_rbac_context,
)
from app.ai.agent.agentic_rag.graph.graph import agentic_rag_graph
from app.ai.orchestration.graph import orchestrator_graph
from langchain_core.messages import HumanMessage

print("=" * 60)
print(f"✅ AI Provider         : {settings.AI_PROVIDER}")
print(f"✅ Embedding Model    : {getattr(settings, 'EMBEDDING_MODEL', 'BAAI/bge-m3')}")
print(f"✅ TEI URL            : {settings.TEI_URL}")
print(f"✅ LLM Base URL       : {settings.LLM_BASE_URL}")
print("=" * 60)

✅ AI Provider         : vllm
✅ Embedding Model    : bge-m3:latest
✅ TEI URL            : http://localhost:8080
✅ LLM Base URL       : http://localhost:11434/v1


In [23]:
embed_svc = get_embedding_service()

test_query = "Quy trình phê duyệt tờ trình mua sắm tài sản trên gAMSPro"
test_batch = [
    "Hạn mức duyệt chi ngân sách hội sở",
    "Hướng dẫn tạo đơn hàng PO và nghiệm thu hàng hóa",
    "Chính sách bảo mật dữ liệu ngân hàng BVBank"
]

print("--- 1. Test Embed Query ---")
vec = await embed_svc.embed_query(test_query)
print(f"Query: '{test_query}'")
print(f"-> Vector Dims: {len(vec)} | Top 5 floats: {[round(x, 4) for x in vec[:10]]}")

print("\n--- 2. Test Embed Cached Query ---")
vec_cached, hit = await embed_svc.embed_query_cached(test_query)
print(f"-> Hit Cache: {hit} (True nếu lần thứ 2 gọi lại cùng query)")

print("\n--- 3. Test Batch Embed Texts ---")
batch_vecs = await embed_svc.embed_texts(test_batch)
print(f"-> Đã sinh embeddings cho {len(batch_vecs)} câu (mỗi vector {len(batch_vecs[0])} dims).")

--- 1. Test Embed Query ---
Query: 'Quy trình phê duyệt tờ trình mua sắm tài sản trên gAMSPro'
-> Vector Dims: 1024 | Top 5 floats: [-0.0391, -0.0703, -0.0208, 0.0176, 0.0217, 0.0003, -0.0059, -0.0257, -0.0231, 0.04]

--- 2. Test Embed Cached Query ---
-> Hit Cache: False (True nếu lần thứ 2 gọi lại cùng query)

--- 3. Test Batch Embed Texts ---
-> Đã sinh embeddings cho 3 câu (mỗi vector 1024 dims).


In [24]:
retriever = get_vector_retriever()

query_test = "quy trình tạo tờ trình mua sắm tài sản"

print(f"🔍 Đang truy vấn ngữ cảnh cho: '{query_test}'...\n")

# Test 1: Truy vấn thông thường (không gán quyền)
res_public = await retriever.retrieve_context(
    query=query_test,
    top_k=3
)

docs = res_public.get("documents", [[]])[0] if res_public.get("documents") else []
citations = res_public.get("citations", [[]])[0] if res_public.get("citations") else []

print(f"✅ Tìm thấy: {len(docs)} đoạn tài liệu liên quan:")
for i, (doc, cit) in enumerate(zip(docs, citations), 1):
    source = cit.get("source") or cit.get("document_name") or "N/A"
    page = cit.get("page") or cit.get("page_number") or ""
    print(f"\n[{i}] Nguồn: {source} (Trang {page})")
    print(f"    Nội dung trích đoạn: {doc}...")

# Test 2: Thử kiểm tra có lọc RBAC
print("\n--- Test RBAC (Role: Admin / Phòng ban: Kế toán) ---")
res_rbac = await retriever.retrieve_context(
    query=query_test,
    top_k=2,
    user_roles="Admin,Purchasing_Specialist",
    user_department="Phòng Kế toán & Mua sắm"
)
docs_rbac = res_rbac.get("documents", [[]])[0] if res_rbac.get("documents") else []
print(f"✅ Tìm thấy với RBAC: {len(docs_rbac)} đoạn tài liệu.")

🔍 Đang truy vấn ngữ cảnh cho: 'quy trình tạo tờ trình mua sắm tài sản'...

✅ Tìm thấy: 3 đoạn tài liệu liên quan:

[1] Nguồn: Slide dao tao Phan mem QLTS GD 3 - APP KIEM KE & PHE DUYET.pptx (Trang 26)
    Nội dung trích đoạn:  đã được tạo trên web.
1. CHỨC NĂNG PHÊ DUYỆT
DUYỆT PHIẾU HỢP ĐỒNG MUA SẮM
1. CHỨC NĂNG PHÊ DUYỆT
Các bước để xét duyệt một tờ trình chủ trương:
Điều kiện: Đăng nhập thành công vào hệ thống và chọn “Chức năng phê duyệt”.
Bước 3: Hiện màn hình chi tiết hợp đồng mua sắm, ấn phê duyệt để tiến hành phê duyệt hợp đồng
DUYỆT PHIẾU HỢP ĐỒNG MUA SẮM
1. CHỨC NĂNG PHÊ DUYỆT
Các bước để xét duyệt một tờ trình chủ trương:
Điều kiện: Đăng nhập thành công vào hệ thống và chọn “Chức năng phê duyệt”.
Bước 4: Hiện Popup xác nhận phê duyệt, nhấn đồng ý để duyệt hợp đồng
Kết quả: Thông báo duyệt thành công
DUYỆT...

[2] Nguồn: Slide dao tao Phan mem QLTS GD 2 - CAP PHE DUYET - KE HOACH - MUA SAM - THANH QUYET TOAN.pptx (Trang 30)
    Nội dung trích đoạn: Đ
1. PHÂN HỆ QUẢN LÝ KẾ HOẠC

In [25]:
# 1. Test liệt kê danh mục tài liệu / quy chế
print("=== 1. Tool: list_policy_categories ===")
cat_result = await list_policy_categories.ainvoke({"category_filter": "Mua sắm"})
try:
    parsed_cats = json.loads(cat_result)
    print(json.dumps(parsed_cats, indent=2, ensure_ascii=False))
except:
    print(cat_result)

# 2. Test tra cứu thông tin chi tiết một tài liệu theo ID hoặc Tên
print("\n=== 2. Tool: get_document_metadata ===")
meta_result = await get_document_metadata.ainvoke({"doc_id": 1})
print(meta_result)

# 3. Test Tool tìm kiếm ngữ cảnh văn bản (tool mà RAG Agent tự gọi)
print("\n=== 3. Tool: search_policy_and_manual_docs ===")
set_rbac_context(user_roles="User", user_department="Hội sở")
tool_search_res = await search_policy_and_manual_docs.ainvoke({
    "query": "hướng dẫn duyệt tờ trình mua sắm",
    "top_k": 2
})
tool_data = json.loads(tool_search_res)
print(f"-> Số chunks trả về: {len(tool_data.get('documents', []))}")
print(f"-> Trích dẫn: {tool_data.get('citations', [])}")


=== 1. Tool: list_policy_categories ===
{
  "categories": [],
  "total_categories": 0
}

=== 2. Tool: get_document_metadata ===
{"error": "Không tìm thấy tài liệu có ID 1"}

=== 3. Tool: search_policy_and_manual_docs ===
-> Số chunks trả về: 2
-> Trích dẫn: [{'source': 'Slide dao tao Phan mem QLTS GD 3 - APP KIEM KE & PHE DUYET.pptx', 'page': 26, 'chunk_id': '2077_25', 'score': 0.01639344262295082}, {'source': 'Slide dao tao Phan mem QLTS GD 3 - APP KIEM KE & PHE DUYET.pptx', 'page': 25, 'chunk_id': '2077_24', 'score': 0.016129032258064516}]


In [26]:
question = "Cho tôi biết quy trình phê duyệt tờ trình mua sắm tài sản trên phần mềm gAMSPro gồm những bước nào?"

# Thiết lập context phân quyền RBAC
set_rbac_context(user_roles="User,Manager", user_department="Phòng Mua sắm")

# Chuẩn bị State theo schema AgenticRagState
initial_state = {
    "messages": [HumanMessage(content=question)],
    "user_query": question,
    "session_id": "test-session-rag-001",
    "user_roles": "User,Manager",
    "user_department": "Phòng Mua sắm",
    "documents": [],
    "citations": [],
    "is_relevant": False,
    "retry_count": 0,
    "final_answer": "",
}

print(f"🚀 Đang gọi Agentic RAG Graph cho câu hỏi:\n'{question}'\n")

# Thực thi Graph
result = await agentic_rag_graph.ainvoke(initial_state)

print("=" * 60)
print("📌 CÂU TRẢ LỜI CUỐI CÙNG (FINAL ANSWER):")
print("=" * 60)
print(result.get("final_answer"))

print("\n" + "=" * 60)
print(f"📚 TRÍCH DẪN NGUỒN ({len(result.get('citations', []))} nguồn):")
print("=" * 60)
for i, cit in enumerate(result.get("citations", []), 1):
    doc_name = cit.get("source") or cit.get("document_name") or cit.get("file_name") or "Tài liệu"
    page = cit.get("page") or cit.get("page_number") or ""
    print(f"[{i}] {doc_name} {f'(Trang {page})' if page else ''}")


🚀 Đang gọi Agentic RAG Graph cho câu hỏi:
'Cho tôi biết quy trình phê duyệt tờ trình mua sắm tài sản trên phần mềm gAMSPro gồm những bước nào?'

📌 CÂU TRẢ LỜI CUỐI CÙNG (FINAL ANSWER):
**1. Danh mục chính sách hiện có:**  
- **Chức năng phê duyệt** (phần mềm gAMSPro): Hướng dẫn quy trình phê duyệt tờ trình mua sắm tài sản.  
- **Điều kiện áp dụng**: Cán bộ có thẩm quyền phê duyệt phải đăng nhập thành công vào hệ thống và chọn chức năng "Phê duyệt".  

**2. Các bước thực hiện:**  
- **Bước 1: Đăng nhập hệ thống**  
  - Cán bộ phê duyệt cần đăng nhập vào hệ thống với tài khoản có quyền truy cập.  

- **Bước 2: Chọn chức năng phê duyệt**  
  - Truy cập vào phần "Chức năng phê duyệt" trong gAMSPro.  

- **Bước 3: Xem chi tiết hợp đồng mua sắm**  
  - Hiện màn hình chi tiết hợp đồng, ấn **phê duyệt** để tiến hành phê duyệt.  

- **Bước 4: Xác nhận phê duyệt**  
  - Hiện popup xác nhận phê duyệt, nhấn **đồng ý** để hoàn tất.  

- **Kết quả:**  
  - Thông báo "Đã phê duyệt thành công" hoặc "X

In [27]:
out_of_domain_query = "Hướng dẫn làm món phở bò Nam Định truyền thống thơm ngon tại nhà"

ood_input = {
    "messages": [HumanMessage(content=out_of_domain_query)],
    "user_query": out_of_domain_query,
    "session_id": "test-ood-001",
    "user_roles": "User",
    "user_department": "Hội sở",
    "documents": [],
    "citations": [],
    "is_relevant": False,
    "retry_count": 0,
    "final_answer": "",
}

print(f"🧪 Test Zero-Hallucination với câu hỏi ngoài phạm vi:\n'{out_of_domain_query}'\n")

ood_result = await agentic_rag_graph.ainvoke(ood_input)

print(f"🎯 Kết quả is_relevant sau cùng : {ood_result.get('is_relevant')}")
print(f"🔁 Số lần retry                : {ood_result.get('retry_count')}")
print(f"💬 Phản hồi nhận được          : {ood_result.get('final_answer')}")


🧪 Test Zero-Hallucination với câu hỏi ngoài phạm vi:
'Hướng dẫn làm món phở bò Nam Định truyền thống thơm ngon tại nhà'

🎯 Kết quả is_relevant sau cùng : False
🔁 Số lần retry                : 0
💬 Phản hồi nhận được          : Tôi là Trợ lý Tra cứu Quy chế, Chính sách & Sổ tay Nghiệp vụ (gAMSPro) của Ngân hàng Bản Việt. Vui lòng cung cấp câu hỏi liên quan đến quy định, chính sách hoặc hướng dẫn sử dụng phần mềm gAMSPro, như:  
- Quy trình mua sắm  
- Quy chế nội bộ ngân hàng  
- Sổ tay hướng dẫn sử dụng phần mềm gAMSPro  

Cảm ơn anh/chị đã sử dụng dịch vụ!


In [28]:
orch_query = "Cho tôi biết hướng dẫn sử dụng và quy trình phê duyệt trên hệ thống gAMSPro"

orch_input = {
    "session_id": "test-orch-rag-01",
    "user_query": orch_query,
    "user_info": {
        "roles": "Purchasing_Officer",
        "department": "Phòng Hành chính Mua sắm",
        "user_id": "tester_01",
    },
    "chat_history": [],
    "messages": [HumanMessage(content=orch_query)],
}

print(f"🌐 Đang gọi Master Orchestrator cho: '{orch_query}'...\n")

orch_result = await orchestrator_graph.ainvoke(orch_input)

print("=" * 60)
print(f"🎯 Intent nhận diện : {orch_result.get('intent')}")
print(f"🛡️ Guardrail Status  : {orch_result.get('guardrail_status')}")
print("=" * 60)
print("💬 Phản hồi (Agent Output):")
print(orch_result.get("agent_output"))

if orch_result.get("citations"):
    print(f"\n📑 Trích dẫn ({len(orch_result['citations'])} tài liệu):")
    for c in orch_result["citations"]:
        print(f"   - {c.get('source') or c.get('document_name')}")


🌐 Đang gọi Master Orchestrator cho: 'Cho tôi biết hướng dẫn sử dụng và quy trình phê duyệt trên hệ thống gAMSPro'...

🎯 Intent nhận diện : None
🛡️ Guardrail Status  : None
💬 Phản hồi (Agent Output):
**1. Danh sách các bước thực hiện**  
**1.1. Bước 1: Đăng nhập hệ thống**  
- Trưởng phòng Hành chính cần đăng nhập vào hệ thống gAMSPro với tài khoản có quyền quản lý.  
- **Nguồn:** Slide dao tao Phan mem QLTS GD 2 - CAP PHE DUYET - KE HOACH - MUA SAM - THANH QUYET TOAN, Trang 40.  

**1.2. Bước 2: Chọn mục Quản lý mua sắm**  
- Trưởng phòng Hành chính chọn mục **Quản lý mua sắm > Đánh giá nhà cung cấp - ĐMMS** trong menu chính.  

**1.3. Bước 3: Tìm kiếm và hiển thị dữ liệu**  
- Nhập thông tin cần tìm kiếm (ví dụ: mã đơn hàng, tên nhà cung cấp) và nhấn nút **Tìm kiếm** để hiển thị dữ liệu.  

**1.4. Bước 4: Chọn dòng dữ liệu và hiển thị chi tiết**  
- Chọn một dòng dữ liệu mong muốn và nhấn nút **Chi tiết** để mở màn hình thông tin chi tiết.  

**1.5. Bước 5: Thực hiện phê duyệt**  
- T

## 🎯 PHẦN TEST FAQ AGENT VÀ MULTILINGUAL RERANKER
Các cell dưới đây giúp kiểm tra toàn diện:
1. **Hybrid Search và BGE Cross-Encoder Reranker**: Lấy Top 10 candidates từ DB và xếp hạng lại Top 3.
2. **FAQ Direct Pipeline (`faq_graph`)**: 1 LLM call trực tiếp, độ trễ < 1s, triệt tiêu hallucination.
3. **Master Orchestrator Integration và Citations**: Đảm bảo Intent `faq` trả về đầy đủ `citations` cho Angular Frontend.

In [29]:
# --- TEST 1: FAQ HYBRID SEARCH VÀ RERANKER ---
from app.ai.agent.faq.services.faq_retriever import retrieve_and_rerank_faqs
from app.ai.rag.reranker import get_reranker

faq_query = "Cho tôi hỏi hotline tổng đài BVBank và thời gian hỗ trợ?"
print(f"🔍 Đang truy xuất và Rerank FAQ cho: '{faq_query}'...")

faq_retrieval_res = await retrieve_and_rerank_faqs(query=faq_query, top_k=3)
print(f"\nKết quả tìm kiếm: found={faq_retrieval_res.get('found')}")

for item in faq_retrieval_res.get('faqs', []):
    print(f"\n[Rank #{item.get('rank')}] Score: {item.get('confidence_score')} | ID: {item.get('faq_id')}")
    print(f"❓ Question: {item.get('question')}")
    print(f"💡 Answer:   {item.get('answer')[:150]}...")

print(f"\n📑 Citations chuẩn hóa ({len(faq_retrieval_res.get('citations', []))} items):")
for cit in faq_retrieval_res.get('citations', []):
    print(f"   - {cit.get('source')} (Score: {cit.get('score')})")


🔍 Đang truy xuất và Rerank FAQ cho: 'Cho tôi hỏi hotline tổng đài BVBank và thời gian hỗ trợ?'...

Kết quả tìm kiếm: found=True

[Rank #1] Score: 1.0 | ID: 1
❓ Question: Thời gian làm việc hành chính tại BVBank quy định như thế nào?
💡 Answer:   ⏰ Thời gian làm việc từ Thứ 2 đến Thứ 6:
- Sáng: 08:00 - 12:00
- Chiều: 13:00 - 17:00
Riêng sáng Thứ 7 áp dụng cho các Chi nhánh/PGD phục vụ giao dịch...

[Rank #2] Score: 0.5 | ID: 7
❓ Question: BVBank có hỗ trợ khám sức khỏe định kỳ cho nhân viên không?
💡 Answer:   ✅ Có. BVBank tổ chức khám sức khỏe định kỳ 1 lần/năm cho toàn thể CBNV chính thức tại các bệnh viện/phòng khám liên kết. Lịch khám cụ thể sẽ được thôn...

[Rank #3] Score: 0.33333 | ID: 9
❓ Question: Nhân viên nữ sinh con được hưởng chế độ thai sản như thế nào?
💡 Answer:   📌 Ngoài chế độ thai sản theo Luật BHXH (6 tháng), BVBank hỗ trợ thêm:
- Trợ cấp sinh con theo quy chế phúc lợi hiện hành
- Ưu tiên bố trí công việc ph...

📑 Citations chuẩn hóa (3 items):
   - FAQ: Thời gian làm v

In [30]:
# --- TEST 2: FAQ AGENT DIRECT PIPELINE (1-Turn LLM) ---
from app.ai.agent.faq.graph.graph import faq_graph
from langchain_core.messages import HumanMessage
import time

test_faq_query = "Làm sao để mở tài khoản thanh toán tại ngân hàng Bản Việt?"
print(f"🤖 FAQ Agent nhận câu hỏi: '{test_faq_query}'...")

start_t = time.time()
faq_res = await faq_graph.ainvoke({
    "user_query": test_faq_query,
    "messages": [HumanMessage(content=test_faq_query)],
    "session_id": "test-faq-session",
})
duration = time.time() - start_t

print("=" * 60)
print(f"⏱️ Thời gian thực thi: {duration:.2f}s (Direct Pipeline vs 3-4s ReAct)")
print("=" * 60)
print("💬 Câu trả lời của FAQ Agent:")
print(faq_res.get("final_answer"))
print("=" * 60)
print(f"📑 Citations đính kèm ({len(faq_res.get('citations', []))} items):")
for cit in faq_res.get('citations', []):
    print(f"   - {cit.get('source')} (ID: {cit.get('chunk_id')})")


🤖 FAQ Agent nhận câu hỏi: 'Làm sao để mở tài khoản thanh toán tại ngân hàng Bản Việt?'...
⏱️ Thời gian thực thi: 6.09s (Direct Pipeline vs 3-4s ReAct)
💬 Câu trả lời của FAQ Agent:
❌ Câu hỏi của bạn không nằm trong danh sách FAQ nội bộ hiện có.  
📞 Bạn nên liên hệ trực tiếp với bộ phận Hành chính - Nhân sự hoặc truy cập website/bank để được hướng dẫn cụ thể về việc mở tài khoản thanh toán.  
ℹ️ Nếu cần hỗ trợ thêm, hãy liên hệ số điện thoại hoặc email được cung cấp trong FAQ #1 nhé! 📞
📑 Citations đính kèm (3 items):
   - FAQ: Bị mất thẻ nhân viên thì cần liên hệ bộ phận nào để làm lại? (ID: faq-14)
   - FAQ: Thời gian làm việc hành chính tại BVBank quy định như thế nào? (ID: faq-1)
   - FAQ: Cách đăng ký cấp phát văn phòng phẩm hàng tháng ra sao? (ID: faq-12)


In [31]:
# --- TEST 3: MASTER ORCHESTRATOR -> FAQ VÀ FRONTEND CITATIONS ---
from app.ai.orchestration.graph import orchestrator_graph

orch_faq_query = "Hotline chăm sóc khách hàng của ngân hàng Bản Việt BVBank là số mấy?"
orch_input_faq = {
    "session_id": "test-orch-faq-01",
    "user_query": orch_faq_query,
    "user_info": {
        "roles": "Staff",
        "department": "Kế toán",
        "user_id": "staff_01",
    },
    "chat_history": [],
    "messages": [HumanMessage(content=orch_faq_query)],
}

print(f"🌐 Master Orchestrator định tuyến: '{orch_faq_query}'...")
orch_faq_res = await orchestrator_graph.ainvoke(orch_input_faq)

print("=" * 60)
print(f"🎯 Intent: {orch_faq_res.get('intent')}")
print("=" * 60)
print("💬 Phản hồi cuối cùng:")
print(orch_faq_res.get("agent_output"))
print("=" * 60)
print(f"📑 Citations trả về cho Frontend Angular ({len(orch_faq_res.get('citations', []))} items):")
for c in orch_faq_res.get('citations', []):
    print(f"   - {c.get('source')} (score: {c.get('score')})")


🌐 Master Orchestrator định tuyến: 'Hotline chăm sóc khách hàng của ngân hàng Bản Việt BVBank là số mấy?'...


[ORCHESTRATOR -> FAQ ERROR] Lỗi khi gọi FAQ Graph: name 'messages' is not defined
Traceback (most recent call last):
  File "c:\2_Company\GSOFT\Enterprice-Chatbot\BVBank-Chatbot\dev_llm_service\app\ai\orchestration\nodes\agent_adapters.py", line 230, in call_faq_agent
    if not messages_list and user_query:
                    ^^^^^^^^
NameError: name 'messages' is not defined


🎯 Intent: None
💬 Phản hồi cuối cùng:
⚠️ Hệ thống tra cứu câu hỏi thường gặp (FAQ) hiện đang gặp sự cố kết nối. Bạn vui lòng thử lại sau ít phút hoặc liên hệ quản trị viên.
📑 Citations trả về cho Frontend Angular (0 items):
